# 02e — Combine all feature tables

Create one analysis table with **one row per `SUBJECT`/`SPELL_IDENTIFIER`**. No surrogate `episode_row_id` is used. If duplicate spell identifiers exist, this notebook keeps the first row in each input table.

In [14]:
%load_ext autoreload
%autoreload 2

In [20]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/app/notebooks/data/synthetic/10000p/")
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]

def keep_unique_rows(df: pd.DataFrame, keys: list[str], name: str) -> pd.DataFrame:
    """Keep the first source row for the given granularity keys."""
    missing = [c for c in keys if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing key columns: {missing}")
    result = df.copy()
    before = len(result)
    result = result.drop_duplicates(keys, keep="first").reset_index(drop=True)
    print(f"{name} ({'-'.join(keys)}): {before:,} rows -> {len(result):,} unique records")
    return result

print(f"Using data from: {DATA_DIR}")


Using data from: /app/notebooks/data/synthetic/10000p


## Load the base episodes and derived features

In [22]:
# 1. Load base episodes
df_base = keep_unique_rows(
    pd.read_csv(
        DATA_DIR / "icare_episodes_anon.csv",
        parse_dates=["ADMISSION_DATE", "DISCHARGE_DATE"],
    ),
    keys=["SUBJECT", "SPELL_IDENTIFIER"],
    name="base episodes",
)

# 2. Define feature files with their respective join keys
feature_sources = {
    # Spell-level features
    "phenotypes": {
        "path": DATA_DIR / "02a_derived_phenotypes_aggregated.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "history": {
        "path": DATA_DIR / "02b_derived_features_phenotypes.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "encounters": {
        "path": DATA_DIR / "02c_derived_features_encounters.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "medications": {
        "path": DATA_DIR / "02d_derived_features_medications.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    "microbiology": {
        "path": DATA_DIR / "02e_derived_features_microbiology.parquet",
        "keys": ["SUBJECT", "SPELL_IDENTIFIER"]
    },
    # Patient-level features (broadcasts across spells)
    "demographics": {
        "path": DATA_DIR / "icare_demographic_anon.parquet",
        "keys": ["SUBJECT"]
    },
}

# 3. Validate existence and load features dynamically
loaded_features = {}
missing = []
for name, meta in feature_sources.items():
    path = meta["path"]
    if not path.exists():
        missing.append(name)
        continue

    # Read parquet or csv automatically based on extension
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    # Standardize keys to string
    for k in meta["keys"]:
        df[k] = df[k].astype(str)
    # Create dictionary
    loaded_features[name] = {
        "df": keep_unique_rows(df, meta["keys"], name),
        "keys": meta["keys"]
    }

if missing:
    raise FileNotFoundError(f"Missing feature files: {', '.join(missing)}. Ensure they are generated first.")


base episodes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
phenotypes (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
history (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
encounters (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
medications (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
microbiology (SUBJECT-SPELL_IDENTIFIER): 10,000 rows -> 10,000 unique records
demographics (SUBJECT): 10,000 rows -> 10,000 unique records


## Merge safely

In [24]:
df_combined = df_base.copy()
df_combined.SUBJECT = df_combined.SUBJECT.astype(str)
df_combined.SPELL_IDENTIFIER = df_combined.SPELL_IDENTIFIER.astype(str)

for name, item in loaded_features.items():
    feature_df = item["df"]
    keys = item["keys"]

    # Drop overlapping columns (except join keys) to avoid duplicate suffixes
    overlapping = [
        c for c in feature_df.columns
        if c in df_combined.columns and c not in KEYS
    ]
    feature_df = feature_df.drop(columns=overlapping)

    # Toggle validation rule based on table grain (spell vs patient)
    validate_rule = "one_to_one" if len(keys) > 1 else "many_to_one"

    df_combined = df_combined.merge(
        feature_df,
        on=keys,
        how="left",
        validate=validate_rule,
    )
    print(f"Merged {name} via {keys}: shape {df_combined.shape}")

assert not df_combined.duplicated(["SUBJECT", "SPELL_IDENTIFIER"]).any(), "Duplicate spells detected post-merge!"
print(f"\nFinal combined dataset ready: {len(df_combined):,} rows, {len(df_combined.columns):,} columns.")


Merged phenotypes via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 84)
Merged history via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 171)
Merged encounters via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 173)
Merged medications via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 174)
Merged microbiology via ['SUBJECT', 'SPELL_IDENTIFIER']: shape (10000, 178)
Merged demographics via ['SUBJECT']: shape (10000, 185)

Final combined dataset ready: 10,000 rows, 185 columns.


In [25]:
output_path = DATA_DIR / "03_derived_analysis_dataset.parquet"
df_combined.to_parquet(output_path, index=False)
print(f"Saved {len(df_combined):,} rows and {len(df_combined.columns):,} columns to {output_path}")
display(df_combined.head())
print(df_combined.columns.tolist())


Saved 10,000 rows and 185 columns to /app/notebooks/data/synthetic/10000p/03_derived_analysis_dataset.parquet


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,time_to_culture_days,time_to_culture_hours,esbl_status,BIRTH_DATE,DEATH_DATE,ETHNICITY,ETHNICITY_DES,GENDER,GENDER_DESC,UPDATE_DT_TM
0,10001,7929089,2005940,4725136,2020-06-10,22:22:00,35,2020-07-03,320,cardiology,...,NaN,NaN,unknown,1981-01-23,None,A,White - British,U,Unknown,2023-03-28
1,10002,8200425,6570033,6896867,2020-03-27,17:31:00,63,2020-03-28,301,gastroenterology,...,NaN,NaN,unknown,1956-11-10,None,L,Any other Asian background,U,Unknown,2023-03-26
2,10003,9931252,5492977,1426665,2023-01-06,00:18:00,89,2023-01-25,301,gastroenterology,...,NaN,NaN,unknown,1965-05-14,None,N,Black or Black British - African,I,Indeterminate,2023-04-04
3,10004,5095152,3730006,3105590,2022-09-14,10:32:00,25,2022-10-12,301,gastroenterology,...,NaN,NaN,unknown,1980-03-11,None,L,Any other Asian background,U,Unknown,2023-05-10
4,10005,7807609,7413177,1104787,2023-09-08,10:55:00,80,2023-09-23,140,oral surgery,...,NaN,NaN,unknown,1977-12-18,None,Z,Not stated,M,Male,2023-07-14


['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_hosp_90d', 'tumbarello_hx_recent_ab